# 14 — PCA, Seleção e Redução de Dimensionalidade

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/14_pca_reducao.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_pca_reducao.pdf)
- [Documentação da área](./README.pt-BR.md)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Explicar autovetores/autovalores, variância explicada, projeções, redução de dimensionalidade, seleção de atributos e vazamento.

PCA encontra direções ortogonais de maior variância dos dados centrados (e frequentemente escalonados); não usa rótulos. Alta variância não significa automaticamente alta relevância preditiva. Seleção de atributos escolhe colunas; PCA constrói combinações. Ambos devem ser ajustados somente no treino quando usados em modelo supervisionado.


## 2. Fundamentos matemáticos

Para matriz centrada $X_c$, matriz de covariância $\Sigma=X_c^\top X_c/(n-1)$. PCA usa autovetores $v_j$ de $\Sigma$ com autovalores $\lambda_j$ decrescentes. Variância explicada proporcional: $\lambda_j/\sum_k\lambda_k$.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
X,y=load_wine(return_X_y=True)
a,b,c,d=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
pipe=make_pipeline(StandardScaler(),PCA(n_components=.95),LogisticRegression(max_iter=1500)).fit(a,c)
pca=pipe.named_steps['pca'];print('Componentes para 95%:',pca.n_components_,'variância acumulada:',round(pca.explained_variance_ratio_.sum(),3))
print('Teste:',round(pipe.score(b,d),3))

Componentes para 95%: 10 variância acumulada: 0.964
Teste: 0.978


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
from sklearn.feature_selection import SelectKBest,f_classif
from sklearn.pipeline import Pipeline
select=Pipeline([('scale',StandardScaler()),('select',SelectKBest(score_func=f_classif,k=5)),('model',LogisticRegression(max_iter=1000))]).fit(a,c)
print('Selecionadas:',select.named_steps['select'].get_support(indices=True),'teste:',round(select.score(b,d),3))

Selecionadas: [ 0  6  9 11 12] teste: 1.0


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: variância acumulada
import numpy as np
ratio=np.array([.55,.25,.12,.08]);print('Componentes necessários para >=80%:',int(np.searchsorted(np.cumsum(ratio),.80)+1))

Componentes necessários para >=80%: 2


## 6. Interpretação, limitações e erros comuns

PCA pode prejudicar interpretação, pois componentes misturam variáveis. O percentual de variância preservada não mede diretamente desempenho preditivo. Se usar PCA em validação cruzada, colocá-lo dentro do pipeline para evitar vazamento.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- PCA é não supervisionado; seleção de atributos pode ser supervisionada.
- Ajustar `PCA` e `SelectKBest` apenas dentro do treino/folds.
- Padronizar atributos com unidades muito distintas.
- Documentar trade-off desempenho × interpretabilidade.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular


**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** variância explicada. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **variância explicada** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **variância explicada** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
